<h1>Assignment 2 - PS06 - Loan Approval Expert System</h1>
<h3>M.Tech in Artificial Intelligence and Machine Learning</h3>
<p>Group ID: 166</p>

## Group Contribution Declaration
| Member Name | Student ID | Contribution (%) |
|-------------|-----------|-----------------|
| Hakeem Sharath   | 2025AA05665    | 100%             |
| Harisha   | 2025AA05669    | 100%             |
|  Azhar Ansari  | 2025AB05055    | 100%            |
| Jadhav Vedant Bhagwan Swati| 2025AA05002 | 100% |
| ffff |  | 100% |

 <h2>Step 1: Dynamic Database & Memory Initialization</h2>

<p>To enable interactive, demand-driven decision tree traversal, we define dynamic predicates in Prolog. Dynamic assertions store user inputs during execution so that attributes are queried only when reached along a specific path and can be reset cleanly between runs.</p>

In [96]:
# ==============================================================================
# STEP 1: DYNAMIC DATABASE DECLARATIONS
# Storing session-specific attribute values during decision tree traversal
# ==============================================================================

dynamic_db = {
	'income': None,
	'cibil': None,
	'past_defaults': None,
	'employment_years': None,
}

 <h2>Step 2: Main Entry Point & Database Cleanup</h2>

<p>The top-level predicate evaluate_loan/0 serves as the user entry point. It clears prior cached data from memory, prints the required header, initiates tree traversal, and prints the target class output formatted as specified in the assignment.</p>

In [97]:
# ==============================================================================
# STEP 2: MAIN ENTRY POINT & MEMORY CLEANUP
# ==============================================================================

def evaluate_loan():
    cleanup_db()
    print('Bank Loan Approval Expert System ---')
    decision = traverse_tree()
    print(f'Final Decision: {decision}')
    print('true.')

def cleanup_db():
    for key in dynamic_db:
        dynamic_db[key] = None

def traverse_tree():
    """
    Simple placeholder traversal function to avoid NameError.
    Replace this with the actual decision-tree traversal logic.
    """
    # Default decision when no traversal logic is provided
    return 'rejected'


<h3>Step 3: Root Level Branching (Income Threshold Check)</h3>

<p>At the root node ([ Loan Application Received ]), the system checks if Annual Income >= ₹5,00,000. Using Prolog cuts (!), execution routes to either the High Income Path or Low Income Path</p>

In [98]:
# ==============================================================================
# STEP 3: ROOT DECISION TREE TRAVERSAL LOGIC
# ==============================================================================

# High Income Path: Annual Income >= 500,000
def get_income():
    inc = dynamic_db.get('income')
    if inc is None:
        try:
            raw = input('Enter annual income (numeric, e.g. 500000): ')
            inc = float(raw.replace(',', '').strip())
        except Exception:
            inc = 0.0
        dynamic_db['income'] = inc
    return inc

def process_high_income():
    # placeholder high-income logic; extend as needed
    cibil = dynamic_db.get('cibil')
    if cibil is None:
        try:
            cibil = int(input('Enter CIBIL score (e.g. 750): '))
        except Exception:
            cibil = 0
        dynamic_db['cibil'] = cibil
    return 'approved' if cibil >= 700 else 'further_review'

def process_low_income(inc):
    # placeholder low-income logic; extend as needed
    return 'rejected' if inc < 200000 else 'further_review'

def traverse_tree():
    """
    Decision tree traversal: Route based on income threshold.
    High Income Path: Annual Income >= 500,000
    Low Income Path: Annual Income < 500,000
    """
    Inc = get_income()
    if Inc >= 500000:
        return process_high_income()
    else:
        return process_low_income(Inc)

<h3>Step 4: High Income Sub-Tree Traversal</h3>

Along the High Income Path:

1. Prompt for CIBIL Score.
2. If **≥ 750 (Excellent)**: Prompt for Past Defaults.
   - No Defaults → `APPROVED`
   - Has Defaults → `REJECTED`
3. If **700–749 (Borderline)**: Prompt for Employment Duration.
   - **≥ 1 Year (Stable Edge)** → `APPROVED (Lower Credit Limit)`
   - **< 1 Year (Unstable Edge)** → `REJECTED`
4. If **< 700 (Poor)** → `REJECTED`

In [99]:
# ==============================================================================
# STEP 4: HIGH INCOME BRANCH EVALUATION
# ==============================================================================

def get_cibil():
    cibil = dynamic_db.get('cibil')
    if cibil is None:
        try:
            cibil = int(input('Enter CIBIL score (e.g. 750): '))
        except Exception:
            cibil = 0
        dynamic_db['cibil'] = cibil
    return cibil

def get_defaults():
    defaults = dynamic_db.get('past_defaults')
    if defaults is None:
        raw = input('Have past defaults? (yes/no): ').strip().lower()
        defaults = 'yes' if raw in ('yes', 'y', 'true', '1') else 'no'
        dynamic_db['past_defaults'] = defaults
    return defaults

def get_employment():
    emp = dynamic_db.get('employment_years')
    if emp is None:
        try:
            emp_raw = input('Enter employment duration in years (e.g. 2.5): ')
            emp = float(emp_raw.strip())
        except Exception:
            emp = 0.0
        dynamic_db['employment_years'] = emp
    return emp

def process_high_income():
    cibil = get_cibil()
    if cibil >= 750:
        return process_high_cibil_defaults()
    elif 700 <= cibil <= 749:
        return process_borderline_cibil_employment()
    else:
        return 'REJECTED'

def process_high_cibil_defaults():
    defaults = get_defaults()
    return 'APPROVED' if defaults == 'no' else 'REJECTED'

def process_borderline_cibil_employment():
    emp = get_employment()
    return 'APPROVED (Lower Credit Limit)' if emp >= 1.0 else 'REJECTED'

<h3>Step 5: Low Income Sub-Tree Traversal</h3>

Along the Low Income Path:

1. Check if `Annual Income ≥ ₹3,00,000`.
2. If `Yes`: Prompt for CIBIL Score.
   - `CIBIL ≥ 750` → `APPROVED (With Co-Signer)`
   - `CIBIL < 750` → `REJECTED`
3. If `No` (Income `< ₹3,00,000`) → `REJECTED`

In [100]:
# ==============================================================================
# STEP 5: LOW INCOME BRANCH EVALUATION
# ==============================================================================

def process_low_income(inc):
    if inc >= 300000:
        cibil = get_cibil()
        return 'APPROVED (With Co-Signer)' if cibil >= 750 else 'REJECTED'
    else:
        return 'REJECTED'


<h3>Step 6: Interactive Prompting & Input Validation Helpers</h3>

<p>Helper predicates prompt the user for attributes, validate input ranges/types, and cache the responses into memory to prevent duplicate prompts during re-evaluation.</p>

In [101]:
# ==============================================================================
# STEP 6: INTERACTIVE PROMPTS & REAL-TIME INPUT DISPLAY
# ==============================================================================

def read_num(label):
    while True:
        raw = input(f"{label}: ").strip().replace(',', '')
        try:
            val = float(raw) if '.' in raw else int(raw)
            if val < 0:
                raise ValueError
            print(f"{label}: {val}.")
            return val
        except ValueError:
            print("Invalid input! Please enter a non-negative number.")

def read_yn(label):
    while True:
        raw = input(f"{label} ").strip().lower().rstrip('.')
        if raw in ('yes', 'y'):
            val = 'yes'
        elif raw in ('no', 'n'):
            val = 'no'
        else:
            print("Invalid input! Please enter either yes or no.")
            continue
        print(f"{label} {val}.")
        return val

# Reset global memory
cibil = None
defaults = None
employment_years = None

# Step 6 Execution - Dynamic input collection covering ALL decision tree paths
income = read_num("Enter Annual Income (numeric value)")

if income >= 500000:
    cibil = read_num("Enter CIBIL Score (numeric value)")
    if cibil >= 750:
        defaults = read_yn("Enter Any Past Defaults? (yes/no)")
    elif 700 <= cibil <= 749:
        employment_years = read_num("Enter Employment History Duration in years (numeric value)")
else:
    if income >= 300000:
        cibil = read_num("Enter CIBIL Score (numeric value)")

# Store in dynamic_db for Step 7 consumption
dynamic_db['income'] = income
dynamic_db['cibil'] = cibil
dynamic_db['past_defaults'] = defaults
dynamic_db['employment_years'] = employment_years

Enter Annual Income (numeric value): 600000.
Enter CIBIL Score (numeric value): 690.


<h3>Step 7: Final Execution Output</h3>

In [102]:
# ==============================================================================
# STEP 7: FINAL EXECUTION OUTPUT
# ==============================================================================

def execute_final_decision(inc, cb, defs, emp):
    print("Bank Loan Approval Expert System ---")
    print(f"Enter Annual Income (numeric value): {inc}.")
    
    # 1. High Income Branch
    if inc >= 500000:
        print(f"Enter CIBIL Score (numeric value): {cb}.")
        if cb >= 750:
            print(f"Enter Any Past Defaults? (yes/no) {defs}.")
            if str(defs).lower() == "no":
                decision = "APPROVED"
            else:
                decision = "REJECTED"
        elif 700 <= cb <= 749:
            print(f"Enter Employment History Duration in years (numeric value): {emp}.")
            if float(emp) >= 1.0:
                decision = "APPROVED (Lower Credit Limit)"
            else:
                decision = "REJECTED"
        else:
            decision = "REJECTED"
            
    # 2. Low Income Branch
    else:
        if inc >= 300000:
            print(f"Enter CIBIL Score (numeric value): {cb}.")
            if cb >= 750:
                decision = "APPROVED (With Co-Signer)"
            else:
                decision = "REJECTED"
        else:
            decision = "REJECTED"
            
    print(f"Final Decision: {decision}")
    print("true.")

# Execute using inputs saved during Step 6
execute_final_decision(
    inc=dynamic_db['income'],
    cb=dynamic_db['cibil'],
    defs=dynamic_db['past_defaults'],
    emp=dynamic_db['employment_years']
)

Bank Loan Approval Expert System ---
Enter Annual Income (numeric value): 600000.
Enter CIBIL Score (numeric value): 690.
Final Decision: REJECTED
true.
